# Zomato Bangalore Restaurant Analytics
## Notebook 3: Visualizations & Insights

Build charts and visualizations around customer segments and online ordering patterns.

---


In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

print('Visualization libraries loaded.')

In [ ]:
# Connect to Database
conn = sqlite3.connect('../zomato_bangalore.db')

def fetch_data(query):
    return pd.read_sql_query(query, conn)

print('Connected to database.')

---
## 1. Online Orders
Looking at whether online ordering correlates with higher ratings.

In [ ]:
# Fetch Online Order vs Rating Data
df_online = fetch_data("""
    SELECT rating, online_order
    FROM restaurants
    WHERE rating IS NOT NULL
""")
df_online['Order Type'] = df_online['online_order'].map({1: 'Accepts Online Orders', 0: 'No Online Orders'})

# Plot
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_online, x='rating', hue='Order Type', fill=True, common_norm=False, palette='crest')
plt.title('Rating Distribution: Online vs Offline Orders', fontsize=16, fontweight='bold')
plt.xlabel('Rating (out of 5)')
plt.ylabel('Density')
plt.axvline(df_online[df_online['online_order']==1]['rating'].mean(), color='teal', linestyle='--', label='Avg (Online)')
plt.axvline(df_online[df_online['online_order']==0]['rating'].mean(), color='navy', linestyle='--', label='Avg (Offline)')
plt.legend()
plt.savefig('../images/online_vs_offline_rating.png', bbox_inches='tight')
plt.show()

> Restaurants with online ordering tend to rate higher (clustered around 3.8-4.0) compared to those without it.

---
## 2. Customer Segments (Budget vs Luxury)

In [ ]:
# Fetch Customer Segment Data
df_segments = fetch_data("""
    WITH Segmented AS (
        SELECT 
            location, approx_cost, rating,
            NTILE(4) OVER (ORDER BY approx_cost) as quartile
        FROM restaurants
        WHERE approx_cost IS NOT NULL AND rating IS NOT NULL
    )
    SELECT 
        location, 
        CASE quartile 
            WHEN 1 THEN 'Budget'
            WHEN 2 THEN 'Mid-Range'
            WHEN 3 THEN 'Premium'
            WHEN 4 THEN 'Luxury'
        END as Segment,
        AVG(rating) as avg_rating,
        COUNT(*) as total_restaurants
    FROM Segmented
    GROUP BY location, quartile
    HAVING total_restaurants > 20
""")

# Interactive Treemap
fig = px.treemap(
    df_segments, 
    path=[px.Constant('Bangalore'), 'Segment', 'location'], 
    values='total_restaurants',
    color='avg_rating',
    color_continuous_scale='RdYlGn',
    title='Restaurant Market Segmentation by Location and Rating'
)
fig.update_layout(margin = dict(t=50, l=25, r=25, b=25))
fig.write_html('../images/segmentation_treemap.html')
fig.show()

> The premium and luxury segments rate higher on average, but budget restaurants make up most of the market — especially in BTM and HSR.

---
## 3. Market Saturation

In [ ]:
# Location Competitiveness
df_loc = fetch_data("""
    SELECT 
        location, 
        COUNT(*) as restaurant_count, 
        AVG(rating) as avg_rating,
        AVG(approx_cost) as avg_cost
    FROM restaurants 
    WHERE location IS NOT NULL AND rating IS NOT NULL
    GROUP BY location
    HAVING restaurant_count > 50
""")

fig = px.scatter(
    df_loc, 
    x='restaurant_count', 
    y='avg_rating', 
    size='avg_cost', 
    color='avg_rating',
    hover_name='location',
    title='Market Saturation: Number of Restaurants vs Average Rating',
    labels={'restaurant_count': 'Number of Restaurants (Density)', 'avg_rating': 'Average Rating'},
    color_continuous_scale='Viridis'
)
fig.add_hline(y=df_loc['avg_rating'].mean(), line_dash="dot", annotation_text="Avg City Rating")
fig.add_vline(x=df_loc['restaurant_count'].mean(), line_dash="dot", annotation_text="Avg Density")
fig.write_html('../images/market_saturation.html')
fig.show()

> Areas in the top-left (few restaurants, high ratings) look like good opportunities. Areas in the bottom-right (lots of restaurants, lower ratings) are already crowded.

---
## 4. Popular Dishes

In [ ]:
df_dishes = fetch_data("SELECT dish FROM restaurant_dishes")
text = ' '.join(df_dishes['dish'].dropna().values)

wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='magma').generate(text)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Most Loved Dishes in Bangalore', fontsize=20, fontweight='bold')
plt.savefig('../images/dishes_wordcloud.png', bbox_inches='tight')
plt.show()

In [ ]:
conn.close()
print('All visualizations generated and saved to /images folder.')